In [ ]:
from pathlib import Path
import sys, importlib
import numpy as np
import pandas as pd

current_path = Path.cwd().resolve()
project_path = None
for path in [current_path, *current_path.parents]:
    if (path / "src").exists() and (path / "official-repository").exists():
        project_path = path
        break
if project_path is None:
    raise FileNotFoundError("Folder proyek (berisi src/ dan official-repository/) tidak ditemukan.")
if str(project_path) not in sys.path:
    sys.path.insert(0, str(project_path))

import src.feature_engineering as fe
import src.possession_model as pm
import src.temporal_analysis as ta
import src.prediction as pr
for module in (fe, pm, ta, pr):
    importlib.reload(module)

print("Project:", project_path)

In [ ]:
TEAM_NAME = "BAJANG_TACTIC"   # GANTI bila nama tim di formulir pendaftaran berbeda
submission_csv = project_path / "submissions" / f"{TEAM_NAME}.csv"

import subprocess, shutil
official_repo = project_path / "official-repository" / "more-than-a-hack-2027"

def run_official(script, *args):
    result = subprocess.run([sys.executable, f"evaluation/{script}", *map(str, args)],
                            cwd=official_repo, capture_output=True, text=True)
    return (result.stdout + result.stderr).strip()

In [ ]:
pr.main(["--team", TEAM_NAME])

In [ ]:
print(run_official("validate_submission.py", submission_csv.resolve()))

In [ ]:
results_dir = official_repo / "evaluation" / "results"
shutil.copy2(submission_csv, results_dir / submission_csv.name)
print(run_official("run_evaluation.py"))
(results_dir / submission_csv.name).unlink(missing_ok=True)

In [ ]:
labels = pd.read_csv(submission_csv)["label"].astype(str).str.strip().str.upper().to_numpy()
graded = pm.get_graded_mask(len(labels))

print("Jumlah baris:", len(labels), "(harus 199319)")
print("Label unik  :", sorted(set(labels)))
print("X di frame dinilai   :", int(np.sum(labels[graded] == "X")), "(sebaiknya 0)")
print("Label non-X di luar babak:", int(np.sum(labels[~graded] != "X")), "(harus 0)")
for half, (s, e) in enumerate(pr.HALVES, start=1):
    print(f"Babak {half}:", ta.summarize_runs(labels, s, e))

In [ ]:
import hashlib
first = hashlib.md5(submission_csv.read_bytes()).hexdigest()

check_csv = project_path / "experiments" / "repro_check.csv"
pr.main(["--team", TEAM_NAME, "--out", str(check_csv)])
second = hashlib.md5(check_csv.read_bytes()).hexdigest()

print("MD5 pertama:", first)
print("MD5 kedua  :", second)
print("IDENTIK" if first == second else "BERBEDA - periksa sumber ketidakpastian!")